# SLM Pivot — Litmus Test (prompted / zero-shot, NO fine-tuning)

This notebook runs the **gating** experiment before any item-bank build or
fine-tuning. It tests whether a *prompted* model can tag a student's chosen
wrong answer (distractor) on an AP Bio MCQ to a **mid-grained misconception**,
and do it **consistently** across repeated runs on novel item stems.

See `docs/litmus_plan.md` for the full plan and the 2x2 interpretation.

| Arm | What it decides |
|-----|-----------------|
| base `Qwen/Qwen3-1.7B` | consistent + accurate ⇒ **no fine-tuning project** |
| frontier (optional) | base fails but frontier works ⇒ **distill into an SLM** |
| neither | ⇒ **fine-tuning justified** (then check labels aren't ambiguous) |

**The bar is CONSISTENCY, not one-shot accuracy.** Runtime → **T4 GPU** for the
base arm. Run top to bottom. Cells are self-contained; the model id is hardcoded.

In [ ]:
import os

REPO = '/content/algebra_error_classifier'
GITHUB = 'https://github.com/gabriel-xiong/algebra_error_classifier.git'

if not os.environ.get('COLAB_GPU'):
    print('WARNING: No GPU detected. Runtime -> Change runtime type -> T4 GPU for the base arm.')

!rm -rf {REPO}
!git clone {GITHUB} {REPO}
os.chdir(REPO)
print('cwd:', os.getcwd())
!ls data/ | grep -i -E 'litmus|misconception'

In [ ]:
# Prompted litmus only needs transformers + torch + accelerate (NO unsloth, NO training).
!pip -q install -U transformers accelerate torch

In [ ]:
# Sanity: confirm the harness and its data are present before running.
!python scripts/litmus_tagging.py --selftest --data data/litmus_apbio_seed.jsonl --runs 3 | head -n 12

## Arm 1 — base Qwen/Qwen3-1.7B (the deployable-SLM candidate)

`--runs 5` drives the consistency metric (modal agreement across 5 samples).
First run downloads the ~1.7B weights (a few minutes).

In [ ]:
!python scripts/litmus_tagging.py \
  --model Qwen/Qwen3-1.7B \
  --data data/litmus_apbio_seed.jsonl \
  --runs 5 \
  --save-predictions outputs/litmus/base_qwen3_1p7b.jsonl

## Arm 2 — frontier model (OPTIONAL)

Only needed if the base 1.7B is **not** consistent. Provide an API key; the
harness auto-selects OpenAI or Anthropic and skips cleanly if neither is set.
This is still **prompted / zero-shot** — no fine-tuning.

In [ ]:
import getpass, os

# Set ONE of these (leave blank to skip). Uncomment the provider you want.
# os.environ['OPENAI_API_KEY'] = getpass.getpass('OPENAI_API_KEY: ')
# os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('ANTHROPIC_API_KEY: ')

if os.environ.get('OPENAI_API_KEY'):
    !pip -q install -U openai
elif os.environ.get('ANTHROPIC_API_KEY'):
    !pip -q install -U anthropic
else:
    print('No frontier key set — the next cell will print a clean skip message.')

In [ ]:
!python scripts/litmus_tagging.py \
  --frontier auto \
  --data data/litmus_apbio_seed.jsonl \
  --runs 5 \
  --save-predictions outputs/litmus/frontier.jsonl

## Read the result (the 2x2)

- **base 1.7B consistent + accurate** → a prompt already does it → **no
  fine-tuning project**; the SLM's value collapses to the *deployment-efficiency*
  story (cheap/on-device tagging at scale), not a capability story.
- **base fails, frontier works** → the project is **distilling** the frontier
  capability into a deployable small model.
- **neither consistent** → **fine-tuning is justified** — then verify the
  misconception labels aren't ambiguous (confident-but-wrong = label problem).

**This gates the pivot.** Do NOT build the full item bank or train until these
results are in — and remember the seed set here is an AUTHORED PLACEHOLDER;
defensible numbers need ~40–50 REAL tagged items (real items are eval-only).

# PART 2 — GENERATION litmus (the PRIMARY v1 task)

Per the mentor-endorsed pivot (`docs/mcat_pivot_spec.md` §12), the primary v1 SLM
task is **conditional generation of misconception-tagged items**: spec → tagged
item. This section gates it. Still **prompted / zero-shot — no fine-tuning.**

**Generate-and-verify (the verifiers ARE the metric):**
- **V1 structural validity** (deterministic) — anti-drift.
- **spec-adherence** — right topic + target misconception(s).
- **V3 tag-fidelity** — an *independent* tagger (`litmus_tagging.predict_tag`)
  re-reads each distractor and must agree with the generator's claimed tag.

**2x2:** base 1.7B reliable ⇒ deployment-efficiency story · base drifts but
frontier reliable ⇒ **distill** (the pivot) · neither ⇒ rework schema/taxonomy.

Independence is enforced: the verifier must be a **different** model than the
generator. A ~30–50-item **human-labeled sample** must anchor verifier↔human
agreement before the fidelity number is trusted (described in the plan; not built).

In [ ]:
# Sanity: dummy generator + dummy verifier (no GPU / no key needed).
!python scripts/litmus_generation.py --selftest --num-specs 6 --runs 1 | head -n 16

## Gen Arm 1 — base Qwen/Qwen3-1.7B generator (the deployable-SLM candidate)

Independent tag-fidelity verifier options (pick one, must DIFFER from the generator):
- `--verifier-frontier auto` (needs a key set in the Part-1 frontier cell), or
- `--verifier-model <a different HF tagger id>`, or
- omit both → tag-fidelity reported as **N/A** (V1 + spec-adherence still run).

In [ ]:
!python scripts/litmus_generation.py \
  --model Qwen/Qwen3-1.7B \
  --verifier-frontier auto \
  --num-specs 20 --runs 2 \
  --save-predictions outputs/litmus/gen_base_qwen3_1p7b.jsonl

## Gen Arm 2 — prompted frontier generator (the teacher / bar to beat)

Needs a frontier key (set in the Part-1 frontier cell). Uses base Qwen3-1.7B as
the independent verifier here so the verifier differs from the frontier generator.
Skips cleanly if no key is set.

In [ ]:
!python scripts/litmus_generation.py \
  --frontier auto \
  --verifier-model Qwen/Qwen3-1.7B \
  --num-specs 20 --runs 1 \
  --save-predictions outputs/litmus/gen_frontier.jsonl

## Read the generation result (the distillation 2x2)

- **base 1.7B validity/fidelity/yield already high** → a prompt suffices; the SLM's
  value is **deployment efficiency** (cheap/on-device generation at scale).
- **base 1.7B DRIFTS (low validity/fidelity) but the frontier is high** → **distill**
  the frontier teacher into a deployable small generator. ← the pivot.
- **neither reliable** → the schema / taxonomy / task needs rework before training.

**This gates the pivot.** Do NOT build the full item bank or fine-tune until these
numbers are in. Remember: the specs here are AUTHORED PLACEHOLDERS; defensible
numbers need a GPU (base 1.7B), an optional frontier API key (teacher + independent
verifier), and a ~30–50-item human-labeled sample anchoring verifier↔human agreement.